In [3]:
import os
import cv2
import csv
import mediapipe as mp
import matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

fileLoc = r'Z:\RCP_Data\IO\CASE_12102025_ETVIM\Video\MediaPipe\cropped'
model_path = "hand_landmarker.task"

if not os.path.exists(model_path):
    import urllib.request
    print("Downloading model...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)


# for adding colors to labels in images
cmap = plt.get_cmap('plasma')
PLASMA_COLORS = []
for i in range(21):
    rgba = cmap(i / 20.0)
    bgr = (int(rgba[2]*255), int(rgba[1]*255), int(rgba[0]*255))
    PLASMA_COLORS.append(bgr)
# for adding the skeleton to image
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]


def draw_hand_on_frame(image, landmarks):
    h, w = image.shape[:2]

    # draw skeleton
    for a, b in HAND_CONNECTIONS:
        xa, ya = int(landmarks[a].x * w), int(landmarks[a].y * h)
        xb, yb = int(landmarks[b].x * w), int(landmarks[b].y * h)
        cv2.line(image, (xa, ya), (xb, yb), (255, 255, 255), 1)

    # draw landmarks
    for i, lm in enumerate(landmarks):
        x, y = int(lm.x * w), int(lm.y * h)
        cv2.circle(image, (x, y), 4, PLASMA_COLORS[i], -1)

# ---- Core Processor ----
def process_video_pipeline(video_path, model_path):
    """
    1. Reads Video
    2. Extract Landmarks -> CSV (Pixel coords, 2 decimal places)
    3. Draw Landmarks -> New Video File
    """

    # -- 1. Setup Output Paths --
    base_name = os.path.splitext(video_path)[0]
    output_csv_path = f"{base_name}_data.csv"
    output_vid_path = f"{base_name}_annotated.mp4"

    # -- 2. Setup MediaPipe --
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_hands=2,  # number of hands to detect
        min_hand_detection_confidence=0.5, # good starting point for grayscale images
        min_hand_presence_confidence=0.5, # good starting point for grayscale images
        min_tracking_confidence=0.5, # good starting point for grayscale images
    )
    landmarker = vision.HandLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return

    # Get properties
    width_float  = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height_float = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    width = int(width_float)
    height = int(height_float)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps: fps = 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


    # CSV Header --- Rex, you may need to modify this to match the output of the DLC
    header = ['frame', 'timestamp_ms', 'hand_index', 'hand_label', 'hand_confidence']
    for i in range(21):
        header.extend([f'x_{i}', f'y_{i}'])

    csv_file = open(output_csv_path, mode='w', newline='')
    writer = csv.writer(csv_file)
    writer.writerow(header)

    # Video Writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vid_writer = cv2.VideoWriter(output_vid_path, fourcc, fps, (width, height))

    print(f"Processing: {os.path.basename(video_path)}")
    print(f"  -> CSV: {os.path.basename(output_csv_path)}")
    print(f"  -> Video: {os.path.basename(output_vid_path)}")

    frame_idx = 0

    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok: break

            frame_idx += 1
            timestamp_ms = int((frame_idx / fps) * 1000)

            # Convert for detection
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Detect
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            # Create a copy for drawing annotations
            annotated_frame = frame_bgr.copy()

            if result.hand_landmarks:
                for i, (hand_lms, handedness) in enumerate(zip(result.hand_landmarks, result.handedness)):

                    # --- A. Data Extraction (CSV) ---
                    category = handedness[0]
                    label = category.category_name
                    score = category.score

                    row = [frame_idx, timestamp_ms, i, label, f"{score:.4f}"]

                    for lm in hand_lms:
                        # Float pixel coordinates
                        val_x = lm.x * width_float
                        val_y = lm.y * height_float
                        # Append with 2 decimal places
                        row.extend([f"{val_x:.2f}", f"{val_y:.2f}"])

                    writer.writerow(row)

                    # --- B. Visualization (Video) ---
                    draw_hand_on_frame(annotated_frame, hand_lms)

            # Write the annotated frame to video file
            vid_writer.write(annotated_frame)

            if frame_idx % 50 == 0:
                print(f"  -> {frame_idx}/{total_frames} frames", end='\r')

    except KeyboardInterrupt:
        print("\n  -> Interrupted by user.")
    finally:
        # Cleanup
        cap.release()
        vid_writer.release()
        csv_file.close()
        landmarker.close()
        print(f"\n  -> Completed: {os.path.basename(video_path)}")

# ---- Main Execution Loop ----

# 1. Find all MP4s
all_files = os.listdir(fileLoc)
mp4_files = [f for f in all_files if f.lower().endswith('.mp4') and 'annotated' not in f] # avoid re-processing outputs

print(f"Found {len(mp4_files)} input video(s).")
print("-" * 60)

# 2. Process each
for i, filename in enumerate(mp4_files):
    full_path = os.path.join(fileLoc, filename)
    process_video_pipeline(full_path, model_path)
    print("-" * 60)

print("Batch processing finished.")

Found 6 input video(s).
------------------------------------------------------------
Processing: 20251210_RCP_Anschutz_I_01_02_session002_rightCam-0000_half.mp4
  -> CSV: 20251210_RCP_Anschutz_I_01_02_session002_rightCam-0000_half_data.csv
  -> Video: 20251210_RCP_Anschutz_I_01_02_session002_rightCam-0000_half_annotated.mp4
  -> 6950/6977 frames
  -> Completed: 20251210_RCP_Anschutz_I_01_02_session002_rightCam-0000_half.mp4
------------------------------------------------------------
Processing: 20251210_RCP_Anschutz_I_01_02_session002_topCam-0000_half.mp4
  -> CSV: 20251210_RCP_Anschutz_I_01_02_session002_topCam-0000_half_data.csv
  -> Video: 20251210_RCP_Anschutz_I_01_02_session002_topCam-0000_half_annotated.mp4
  -> 6950/6977 frames
  -> Completed: 20251210_RCP_Anschutz_I_01_02_session002_topCam-0000_half.mp4
------------------------------------------------------------
Processing: 20251210_RCP_Anschutz_I_01_03_session002_rightCam-0000_half.mp4
  -> CSV: 20251210_RCP_Anschutz_I_01_03